In [ ]:
import time

import numpy as np
import requests
import pandas as pd
from io import StringIO

BASE_URL = "https://sportsbookreviewsonline.com/scoresoddsarchives/nfl-odds-{start}-{end}"

# season start years covered: 2007-08 season through 2021-22 season
SEASON_START_YEARS = list(range(2007, 2022))


def season_url(start_year: int) -> str:
    """Build the archive URL for the season beginning in `start_year`,
    e.g. season_url(2021) -> ".../nfl-odds-2021-22"."""
    end_year = start_year + 1
    return BASE_URL.format(start=start_year, end=f"{end_year % 100:02d}")

## Functions

In [ ]:
def fetch_season_table(url: str) -> pd.DataFrame:
    """fetch the page and pull out the raw odds table."""
    headers = {"User-Agent": "Mozilla/5.0"}
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()

    tables = pd.read_html(StringIO(resp.text))
    # the odds table is the largest table on the page
    raw = max(tables, key=len)

    # site renders the header as a normal first data row (no <th>), so
    # pandas assigns 0..N as column names -- promote row 0 to headers.
    if list(raw.columns) == list(range(raw.shape[1])) or all(
        str(c).isdigit() for c in raw.columns
    ):
        raw.columns = [str(c).strip() for c in raw.iloc[0]]
        raw = raw.iloc[1:].reset_index(drop=True)
    else:
        raw.columns = [str(c).strip() for c in raw.columns]

    return raw


def _to_num(x):
    """Convert 'pk' (pick'em) to 0.0, blanks to NaN, else float."""
    if pd.isna(x):
        return float("nan")
    x = str(x).strip().lower()
    if x in ("", "nl"):
        return float("nan")
    if x == "pk":
        return 0.0
    try:
        return float(x)
    except ValueError:
        return float("nan")


def parse_games(raw: pd.DataFrame, season_start_year: int) -> pd.DataFrame:
    """pair away/home rows into one game per row, and split the
    dual-purpose Open/Close columns into spread_line vs total_line.

    `season_start_year` is the year the season kicked off in (e.g. 2021
    for the 2021-22 season) -- it's needed to resolve the Date column
    into a real year, since the site only encodes month/day.
    """
    df = raw.copy()

    rename_map = {}
    for c in df.columns:
        cl = c.lower()
        if cl == "date":
            rename_map[c] = "date"
        elif cl == "rot":
            rename_map[c] = "rot"
        elif cl == "vh":
            rename_map[c] = "vh"
        elif cl == "team":
            rename_map[c] = "team"
        elif cl == "final":
            rename_map[c] = "final"
        elif cl == "open":
            rename_map[c] = "open"
        elif cl == "close":
            rename_map[c] = "close"
        elif cl == "ml":
            rename_map[c] = "ml"
        elif cl == "2h":
            rename_map[c] = "second_half"
    df = df.rename(columns=rename_map)

    keep = ["date", "rot", "vh", "team", "final", "open", "close", "ml"]
    df = df[[c for c in keep if c in df.columns]].copy()

    df["rot"] = pd.to_numeric(df["rot"], errors="coerce")
    df["final"] = pd.to_numeric(df["final"], errors="coerce")
    df["open_num"] = df["open"].apply(_to_num)
    df["close_num"] = df["close"].apply(_to_num)
    df["ml_num"] = pd.to_numeric(df["ml"], errors="coerce")

    # Date comes in as an integer mmdd with no leading zero on the month
    # and no year at all (e.g. 906 = Sep 6, 1216 = Dec 16, 203 = Feb 3).
    # month/day fall out of a simple divmod by 100 regardless of digit
    # count. The year isn't in the data at all, so it's inferred from
    # which half of the season a game falls in: Sep-Dec belong to the
    # season's start year, Jan-Jun (playoffs / Super Bowl) belong to the
    # following year.
    date_num = pd.to_numeric(df["date"], errors="coerce")
    month = date_num // 100
    day = date_num % 100
    year = np.where(month >= 7, season_start_year, season_start_year + 1)
    df["date"] = pd.to_datetime(
        pd.DataFrame({"year": year, "month": month, "day": day}), errors="coerce"
    )

    # rows already come pre-interleaved (away, home, away, home, ...) in
    # the source table -- do NOT sort by rot globally, since rotation
    # numbers reset every week and a global sort scrambles the pairing.
    df = df.dropna(subset=["rot"]).reset_index(drop=True)

    games = []
    i = 0
    while i < len(df) - 1:
        away = df.iloc[i]
        home = df.iloc[i + 1]

        # basic sanity check: rotation numbers should be consecutive (paired)
        if int(home["rot"]) - int(away["rot"]) != 1:
            i += 1
            continue

        # disambiguate spread vs total: the total is the larger magnitude
        # number and is (usually) identical/near-identical on both rows;
        # the spread is the smaller number and differs between rows.
        def split_spread_total(a_val, h_val):
            pair = [a_val, h_val]
            total = max(pair, key=lambda v: abs(v) if pd.notna(v) else -1)
            spread = min(pair, key=lambda v: abs(v) if pd.notna(v) else float("inf"))
            return spread, total

        open_spread, open_total = split_spread_total(away["open_num"], home["open_num"])
        close_spread, close_total = split_spread_total(away["close_num"], home["close_num"])

        games.append({
            "date": away["date"],
            "away_team": away["team"],
            "home_team": home["team"],
            "away_score": away["final"],
            "home_score": home["final"],
            "away_ml": away["ml_num"],
            "home_ml": home["ml_num"],
            "open_spread": open_spread,
            "close_spread": close_spread,
            "open_total": open_total,
            "close_total": close_total,
        })
        i += 2

    return pd.DataFrame(games)

## Pulling data

In [ ]:
all_games = []
for start_year in SEASON_START_YEARS:
    season_label = f"{start_year}-{str(start_year + 1)[-2:]}"
    url = season_url(start_year)
    print(f"Fetching {season_label} ({url}) ...")
    try:
        raw = fetch_season_table(url)
        games = parse_games(raw, season_start_year=start_year)
    except Exception as exc:
        print(f"  FAILED: {exc}")
        continue
    games["season"] = season_label
    all_games.append(games)
    print(f"  {len(games)} games")
    time.sleep(1)  # be polite to the server between requests

all_games_df = pd.concat(all_games, ignore_index=True)
all_games_df.shape

In [ ]:
out_path = "nfl_odds_2007-2022.csv"
all_games_df.to_csv(out_path, index=False)
print(f"saved {len(all_games_df)} games ({all_games_df['season'].nunique()} seasons) to {out_path}")
all_games_df.head(10)